In [1]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile
import os

zip_path = '/content/drive/MyDrive/skin.zip'
extract_path = '/content/skin_dataset'

print("Extracting ZIP file...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Extraction complete!")

Mounted at /content/drive
Extracting ZIP file...
✅ Extraction complete!


In [ ]:

for root, dirs, files in os.walk(extract_path):
    level = root.replace(extract_path, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}📁 {os.path.basename(root)}/')
    if level < 2:  # Show files only 2 levels deep
        for f in files[:5]:  # Show first 5 files
            print(f'{indent}  📄 {f}')
        if len(files) > 5:
            print(f'{indent}  ... and {len(files)-5} more files')

In [ ]:
import pandas as pd

# Search for any CSV files
csv_files = []
for root, dirs, files in os.walk(extract_path):
    for f in files:
        if f.endswith('.csv'):
            csv_files.append(os.path.join(root, f))

if csv_files:
    print("✅ Found CSV files:")
    for c in csv_files:
        print(f"  → {c}")

    # Load and preview the first one
    df = pd.read_csv(csv_files[0])
    print(f"\n📊 Shape: {df.shape}")
    print(df.head())
    print(f"\n📌 Columns: {list(df.columns)}")
else:
    print("⚠️ No CSV found — dataset likely organized in folders by class")

In [ ]:
from collections import defaultdict

image_counts = defaultdict(int)
image_extensions = {'.jpg', '.jpeg', '.png', '.bmp'}

for root, dirs, files in os.walk(extract_path):
    for f in files:
        if os.path.splitext(f)[1].lower() in image_extensions:
            folder_name = os.path.basename(root)
            image_counts[folder_name] += 1

print("🖼️ Image counts per folder:")
total = 0
for folder, count in sorted(image_counts.items()):
    print(f"  {folder}: {count} images")
    total += count

print(f"\n✅ Total images found: {total}")

In [ ]:
label_cols = ['MEL', 'NV', 'BCC', 'AKIEC', 'BKL', 'DF', 'VASC']

disease_names = {
    'MEL'  : 'Melanoma',
    'NV'   : 'Melanocytic Nevi',
    'BCC'  : 'Basal Cell Carcinoma',
    'AKIEC': 'Actinic Keratosis',
    'BKL'  : 'Benign Keratosis',
    'DF'   : 'Dermatofibroma',
    'VASC' : 'Vascular Lesion'
}

# Convert one-hot → single label
df['label']     = df[label_cols].idxmax(axis=1)
df['label_idx'] = df[label_cols].values.argmax(axis=1)
df['img_path']  = df['image'].apply(
    lambda x: f'/content/skin_dataset/images/{x}.jpg'
)

# Verify all image files exist
missing = df[~df['img_path'].apply(os.path.exists)]
print(f"✅ Labels prepared")
print(f"   Total samples  : {len(df)}")
print(f"   Missing images : {len(missing)}")
print(f"\n📊 Class distribution:")
print(df['label'].value_counts())

In [ ]:
import matplotlib.pyplot as plt

class_counts = df['label'].value_counts()

plt.figure(figsize=(10, 5))
bars = plt.bar(
    [disease_names[c] for c in class_counts.index],
    class_counts.values,
    color='steelblue', edgecolor='black'
)
plt.xticks(rotation=30, ha='right')
plt.title('Class Distribution — HAM10000', fontsize=14)
plt.ylabel('Number of Images')

for bar, val in zip(bars, class_counts.values):
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 30,
             str(val), ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150)
plt.show()

In [ ]:
import matplotlib.image as mpimg

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, code in enumerate(label_cols):
    sample_path = df[df['label'] == code]['img_path'].values[0]
    img = mpimg.imread(sample_path)
    axes[i].imshow(img)
    axes[i].set_title(f"{disease_names[code]}\n({code})", fontsize=10)
    axes[i].axis('off')

for j in range(len(label_cols), len(axes)):
    axes[j].axis('off')

plt.suptitle('Sample Image Per Disease Class', fontsize=14)
plt.tight_layout()
plt.savefig('sample_images.png', dpi=150)
plt.show()

In [ ]:
!pip install torchvision -q

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from PIL import Image
import numpy as np
import seaborn as sns

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Using device: {device}")

In [ ]:
# 70% train | 15% val | 15% test (stratified)
train_df, temp_df = train_test_split(
    df, test_size=0.30,
    stratify=df['label_idx'], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50,
    stratify=temp_df['label_idx'], random_state=42
)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f"✅ Train : {len(train_df)}")
print(f"   Val   : {len(val_df)}")
print(f"   Test  : {len(test_df)}")

In [ ]:
IMG_SIZE = 224

train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

print("✅ Transforms ready")

In [ ]:
class SkinDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df        = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        image = Image.open(row['img_path']).convert('RGB')
        label = int(row['label_idx'])
        if self.transform:
            image = self.transform(image)
        return image, label

BATCH_SIZE = 32

train_dataset = SkinDataset(train_df, train_transforms)
val_dataset   = SkinDataset(val_df,   val_transforms)
test_dataset  = SkinDataset(test_df,  val_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)

print(f"✅ DataLoaders ready")
print(f"   Train batches : {len(train_loader)}")
print(f"   Val batches   : {len(val_loader)}")
print(f"   Test batches  : {len(test_loader)}")

In [ ]:
counts = df['label_idx'].value_counts().sort_index().values
weights = 1.0 / counts
class_weights = torch.FloatTensor(weights / weights.sum()).to(device)

print("📊 Class weights (higher = rarer class):")
for code, w in zip(label_cols, class_weights):
    print(f"  {code:6s} | {disease_names[code]:25s} | weight: {w:.4f}")

In [ ]:
NUM_CLASSES = 7

def build_model():
    model = models.efficientnet_b3(weights='IMAGENET1K_V1')

    # Freeze all base layers
    for param in model.parameters():
        param.requires_grad = False

    # Custom classifier head
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, 256),
        nn.ReLU(),
        nn.Dropout(p=0.2),
        nn.Linear(256, NUM_CLASSES)
    )
    return model

model = build_model().to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"✅ EfficientNet-B3 ready")
print(f"   Total params    : {total_params:,}")
print(f"   Trainable params: {trainable_params:,}")

In [ ]:
EPOCHS_HEAD = 5

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.classifier.parameters(), lr=1e-3)

history = {'train_loss': [], 'val_loss': [],
           'train_acc' : [], 'val_acc' : []}

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, correct, total = 0, 0, 0

    with torch.set_grad_enabled(train):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss    = criterion(outputs, labels)

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * images.size(0)
            _, preds    = outputs.max(1)
            correct    += preds.eq(labels).sum().item()
            total      += images.size(0)

    return total_loss / total, correct / total

print("🚀 Phase 1: Training classifier head only...")
for epoch in range(EPOCHS_HEAD):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    vl_loss, vl_acc = run_epoch(val_loader,   train=False)

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(vl_acc)

    print(f"  Epoch {epoch+1}/{EPOCHS_HEAD} | "
          f"Train Loss: {tr_loss:.4f}  Acc: {tr_acc:.4f} | "
          f"Val Loss: {vl_loss:.4f}  Acc: {vl_acc:.4f}")

In [ ]:
EPOCHS_FINE = 10

# Unfreeze all layers
for param in model.parameters():
    param.requires_grad = True

optimizer = optim.Adam(model.parameters(), lr=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_FINE)

best_val_acc = 0.0
print("🔓 Phase 2: Fine-tuning full model...")

for epoch in range(EPOCHS_FINE):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    vl_loss, vl_acc = run_epoch(val_loader,   train=False)
    scheduler.step()

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(vl_acc)

    # Save best model
    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save(model.state_dict(), '/content/best_model.pth')
        print(f"  💾 Best model saved! Val Acc: {vl_acc:.4f}")

    print(f"  Epoch {epoch+1}/{EPOCHS_FINE} | "
          f"Train Loss: {tr_loss:.4f}  Acc: {tr_acc:.4f} | "
          f"Val Loss: {vl_loss:.4f}  Acc: {vl_acc:.4f}")

print(f"\n✅ Best Val Accuracy: {best_val_acc:.4f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history['train_loss'], label='Train Loss', color='blue')
ax1.plot(history['val_loss'],   label='Val Loss',   color='orange')
ax1.axvline(x=EPOCHS_HEAD - 0.5, color='gray',
            linestyle='--', label='Fine-tune start')
ax1.set_title('Loss Curve'); ax1.set_xlabel('Epoch')
ax1.legend(); ax1.grid(True)

ax2.plot(history['train_acc'], label='Train Acc', color='blue')
ax2.plot(history['val_acc'],   label='Val Acc',   color='orange')
ax2.axvline(x=EPOCHS_HEAD - 0.5, color='gray',
            linestyle='--', label='Fine-tune start')
ax2.set_title('Accuracy Curve'); ax2.set_xlabel('Epoch')
ax2.legend(); ax2.grid(True)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

In [ ]:
# Load best model
model.load_state_dict(torch.load('/content/best_model.pth'))
model.eval()

all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images  = images.to(device)
        outputs = model(images)
        _, preds = outputs.max(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

print("📋 Classification Report:")
print(classification_report(
    all_labels, all_preds,
    target_names=[disease_names[c] for c in label_cols]
))

# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=[disease_names[c] for c in label_cols],
            yticklabels=[disease_names[c] for c in label_cols])
plt.title('Confusion Matrix')
plt.ylabel('True Label'); plt.xlabel('Predicted Label')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
import shutil

shutil.copy(
    '/content/best_model.pth',
    '/content/drive/MyDrive/skin_disease_best_model.pth'
)

print("✅ Model saved to Google Drive!")
print(f"   Best Validation Accuracy : {best_val_acc:.4f}")
print(f"   Classes                  : {label_cols}")

In [ ]:
!pip install grad-cam -q
print("✅ grad-cam installed")

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
import cv2

# Load best model (skip if already loaded)
model.load_state_dict(torch.load('/content/best_model.pth'))
model.eval()
model.to(device)

# ✅ Target the last conv layer of EfficientNet-B3
target_layer = [model.features[-1]]

cam = GradCAM(model=model, target_layers=target_layer)
print("✅ Grad-CAM ready")

In [ ]:
def gradcam_single(img_path, true_label=None):
    """Generate Grad-CAM heatmap for a single image"""

    # --- Preprocess ---
    preprocess = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ])

    raw_img = Image.open(img_path).convert('RGB').resize((224, 224))
    rgb_img = np.array(raw_img) / 255.0          # for overlay (float 0-1)
    input_tensor = preprocess(Image.open(img_path).convert('RGB')).unsqueeze(0).to(device)

    # --- Predict ---
    with torch.no_grad():
        output = model(input_tensor)
        probs  = torch.softmax(output, dim=1)[0]
        pred_idx = probs.argmax().item()
        pred_conf = probs[pred_idx].item()

    pred_label = label_cols[pred_idx]

    # --- Generate CAM ---
    targets = [ClassifierOutputTarget(pred_idx)]
    grayscale_cam = cam(input_tensor=input_tensor, targets=targets)[0]
    visualization = show_cam_on_image(rgb_img.astype(np.float32),
                                      grayscale_cam, use_rgb=True)

    # --- Plot ---
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].imshow(raw_img)
    axes[0].set_title('Original Image', fontsize=12)
    axes[0].axis('off')

    axes[1].imshow(grayscale_cam, cmap='jet')
    axes[1].set_title('Grad-CAM Heatmap', fontsize=12)
    axes[1].axis('off')

    axes[2].imshow(visualization)
    title = f"Predicted: {disease_names[pred_label]}\nConfidence: {pred_conf:.2%}"
    if true_label:
        match = "✅" if pred_label == true_label else "❌"
        title += f"\nTrue: {disease_names[true_label]} {match}"
    axes[2].set_title(title, fontsize=11)
    axes[2].axis('off')

    plt.suptitle('Grad-CAM Explainability', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('gradcam_single.png', dpi=150, bbox_inches='tight')
    plt.show()

    return pred_label, pred_conf

# ✅ Test on one image from test set
sample_row = test_df.sample(1).iloc[0]
gradcam_single(sample_row['img_path'], true_label=sample_row['label'])

In [ ]:
def gradcam_grid():
    """Show Grad-CAM for one sample from each disease class"""

    fig, axes = plt.subplots(7, 3, figsize=(14, 28))
    preprocess = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ])

    for row_idx, code in enumerate(label_cols):
        # Pick one sample of this class from test set
        sample = test_df[test_df['label'] == code].sample(1).iloc[0]
        img_path = sample['img_path']

        raw_img = Image.open(img_path).convert('RGB').resize((224, 224))
        rgb_img = np.array(raw_img) / 255.0
        input_tensor = preprocess(Image.open(img_path).convert('RGB')).unsqueeze(0).to(device)

        # Predict
        with torch.no_grad():
            output    = model(input_tensor)
            probs     = torch.softmax(output, dim=1)[0]
            pred_idx  = probs.argmax().item()
            pred_conf = probs[pred_idx].item()

        pred_code = label_cols[pred_idx]

        # Grad-CAM
        targets        = [ClassifierOutputTarget(pred_idx)]
        grayscale_cam  = cam(input_tensor=input_tensor, targets=targets)[0]
        visualization  = show_cam_on_image(rgb_img.astype(np.float32),
                                           grayscale_cam, use_rgb=True)

        # Plot row
        axes[row_idx, 0].imshow(raw_img)
        axes[row_idx, 0].set_ylabel(disease_names[code], fontsize=9, rotation=90)
        axes[row_idx, 0].axis('off')

        axes[row_idx, 1].imshow(grayscale_cam, cmap='jet')
        axes[row_idx, 1].axis('off')

        match = "✅" if pred_code == code else "❌"
        axes[row_idx, 2].imshow(visualization)
        axes[row_idx, 2].set_title(
            f"{match} {disease_names.get(pred_code, pred_code)}\n({pred_conf:.2%})",
            fontsize=8
        )
        axes[row_idx, 2].axis('off')

    # Column headers
    axes[0, 0].set_title('Original',      fontsize=11, fontweight='bold')
    axes[0, 1].set_title('Heatmap',       fontsize=11, fontweight='bold')
    axes[0, 2].set_title('Prediction',    fontsize=11, fontweight='bold')

    plt.suptitle('Grad-CAM — One Sample Per Disease Class',
                 fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig('gradcam_grid.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✅ Saved as gradcam_grid.png")

gradcam_grid()

In [ ]:
def predict_with_confidence(img_path):
    """Show full confidence scores across all 7 classes"""

    preprocess = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ])

    input_tensor = preprocess(
        Image.open(img_path).convert('RGB')
    ).unsqueeze(0).to(device)

    with torch.no_grad():
        probs = torch.softmax(model(input_tensor), dim=1)[0].cpu().numpy()

    pred_idx = probs.argmax()

    # Plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

    # Image
    ax1.imshow(Image.open(img_path).convert('RGB'))
    ax1.set_title('Input Image'); ax1.axis('off')

    # Confidence bars
    colors = ['crimson' if i == pred_idx else 'steelblue' for i in range(7)]
    bars   = ax2.barh([disease_names[c] for c in label_cols],
                       probs, color=colors, edgecolor='black')
    ax2.set_xlim(0, 1)
    ax2.set_xlabel('Confidence')
    ax2.set_title('Prediction Confidence per Class')

    for bar, prob in zip(bars, probs):
        ax2.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                 f'{prob:.2%}', va='center', fontsize=9)

    plt.tight_layout()
    plt.savefig('confidence_chart.png', dpi=150)
    plt.show()

# Test on a random sample
sample_path = test_df.sample(1).iloc[0]['img_path']
predict_with_confidence(sample_path)

In [ ]:
!pip install streamlit pyngrok -q
print("✅ Streamlit installed")

In [ ]:
app_code = \'\'\'
import streamlit as st
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.models as models
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")

# -- Config ------------------------------------------------
st.set_page_config(
    page_title="Skin Disease Detector",
    page_icon="🩺",
    layout="wide"
)

LABEL_COLS = ["MEL", "NV", "BCC", "AKIEC", "BKL", "DF", "VASC"]

DISEASE_INFO = {
    "MEL": {
        "name"       : "Melanoma",
        "severity"   : "🔴 High Risk",
        "description": "A serious form of skin cancer that develops in melanocytes. Early detection is critical.",
        "advice"     : "See a dermatologist immediately for biopsy."
    },
    "NV": {
        "name"       : "Melanocytic Nevi",
        "severity"   : "🟢 Low Risk",
        "description": "Common benign moles formed by clusters of pigmented cells.",
        "advice"     : "Monitor for changes in size, shape, or color. Usually harmless."
    },
    "BCC": {
        "name"       : "Basal Cell Carcinoma",
        "severity"   : "🟠 Moderate Risk",
        "description": "The most common type of skin cancer. Rarely spreads but needs treatment.",
        "advice"     : "Consult a dermatologist for surgical or topical treatment."
    },
    "AKIEC": {
        "name"       : "Actinic Keratosis",
        "severity"   : "🟠 Moderate Risk",
        "description": "Rough, scaly patches caused by years of sun exposure. Can become cancerous.",
        "advice"     : "Seek medical evaluation. Can be treated with cryotherapy or creams."
    },
    "BKL": {
        "name"       : "Benign Keratosis",
        "severity"   : "🟢 Low Risk",
        "description": "Non-cancerous skin growths including seborrheic keratoses and solar lentigines.",
        "advice"     : "Usually harmless. Remove only if causing discomfort."
    },
    "DF": {
        "name"       : "Dermatofibroma",
        "severity"   : "🟢 Low Risk",
        "description": "A common benign skin growth, often appearing on the legs.",
        "advice"     : "No treatment needed unless bothersome."
    },
    "VASC": {
        "name"       : "Vascular Lesion",
        "severity"   : "🟡 Low-Moderate Risk",
        "description": "Includes cherry angiomas and other benign blood-vessel growths in the skin.",
        "advice"     : "Generally harmless. Consult a doctor if it changes rapidly or bleeds."
    }
}

# -- Model loading (cached so it only happens once) --------
@st.cache_resource
def load_model():
    model = models.efficientnet_b3(weights=None)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, 256),
        nn.ReLU(),
        nn.Dropout(p=0.2),
        nn.Linear(256, len(LABEL_COLS))
    )
    model.load_state_dict(
        torch.load("skin_disease_best_model.pth", map_location="cpu")
    )
    model.eval()
    return model

@st.cache_resource
def get_cam(_model):
    target_layer = [_model.features[-1]]
    return GradCAM(model=_model, target_layers=target_layer)

model = load_model()
cam = get_cam(model)

preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# -- UI ------------------------------------------------------
st.title("🩺 Skin Disease Detector")
st.write(
    "Upload a dermoscopic skin lesion image to get a prediction across "
    "7 common skin conditions, with a Grad-CAM explanation of what the "
    "model focused on."
)
st.warning(
    "This tool is for educational/demo purposes only and is **not** a "
    "substitute for professional medical diagnosis."
)

uploaded_file = st.file_uploader(
    "Upload a skin lesion image", type=["jpg", "jpeg", "png"]
)

if uploaded_file is not None:
    raw_img = Image.open(uploaded_file).convert("RGB").resize((224, 224))
    rgb_img = np.array(raw_img) / 255.0
    input_tensor = preprocess(Image.open(uploaded_file).convert("RGB")).unsqueeze(0)

    with torch.no_grad():
        output = model(input_tensor)
        probs = torch.softmax(output, dim=1)[0].numpy()

    pred_idx = int(probs.argmax())
    pred_code = LABEL_COLS[pred_idx]
    pred_conf = probs[pred_idx]
    info = DISEASE_INFO[pred_code]

    targets = [ClassifierOutputTarget(pred_idx)]
    grayscale_cam = cam(input_tensor=input_tensor, targets=targets)[0]
    visualization = show_cam_on_image(rgb_img.astype(np.float32), grayscale_cam, use_rgb=True)

    col1, col2, col3 = st.columns(3)
    with col1:
        st.image(raw_img, caption="Original Image", use_column_width=True)
    with col2:
        st.image(grayscale_cam, caption="Grad-CAM Heatmap", clamp=True, use_column_width=True)
    with col3:
        st.image(visualization, caption="Overlay", use_column_width=True)

    st.subheader(f"Prediction: {info['name']} ({pred_conf:.1%} confidence)")
    st.write(f"**Severity:** {info['severity']}")
    st.write(f"**Description:** {info['description']}")
    st.write(f"**Advice:** {info['advice']}")

    st.subheader("Confidence across all classes")
    fig, ax = plt.subplots(figsize=(8, 4))
    colors = ["crimson" if i == pred_idx else "steelblue" for i in range(len(LABEL_COLS))]
    ax.barh([DISEASE_INFO[c]["name"] for c in LABEL_COLS], probs, color=colors)
    ax.set_xlim(0, 1)
    ax.set_xlabel("Confidence")
    st.pyplot(fig)
\'\'\'

with open("/content/app.py", "w") as f:
    f.write(app_code)

print("✅ app.py written to /content/app.py")


In [ ]:
import shutil
from pyngrok import ngrok
from getpass import getpass
import time

# Copy model
shutil.copy(
    '/content/drive/MyDrive/skin_disease_best_model.pth',
    '/content/skin_disease_best_model.pth'
)

# Enter your ngrok auth token securely (never hardcode tokens in a notebook
# you plan to push to GitHub!). Get one free at https://dashboard.ngrok.com
ngrok_token = getpass("Paste your ngrok auth token: ")
ngrok.set_auth_token(ngrok_token)

# Kill existing tunnels
ngrok.kill()

# Launch app
!streamlit run /content/app.py --server.port 8501 &>/content/streamlit.log &

time.sleep(4)

public_url = ngrok.connect(8501)
print(f"\n✅ App is LIVE at: {public_url}")
print("   Open that URL in your browser!")


In [ ]:
# Cell 29 — Download everything needed
from google.colab import files
import shutil, os

# 1. Download app.py
files.download('/content/app.py')

# 2. Download best model
files.download('/content/skin_disease_best_model.pth')

print("✅ Download started for both files!")

In [ ]:
from google.colab import files
import shutil, os

os.makedirs('/content/demo_images', exist_ok=True)

for code in label_cols:
    sample = test_df[test_df['label'] == code].iloc[0]
    dst = f'/content/demo_images/{code}_{os.path.basename(sample["img_path"])}'
    shutil.copy(sample['img_path'], dst)

shutil.make_archive('/content/demo_images', 'zip', '/content/demo_images')
files.download('/content/demo_images.zip')
print("✅ 7 demo images downloaded!")

NameError: name 'label_cols' is not defined